# Paper Citation Graph — CR → Network → Analysis → Viz

**Pipeline:** WoS `CR` parsing → internal citation matching (DOI/key) → directed graph metrics → cross-country flow → optional topic coherence → frontier gap → summary manifest.

**Outputs** → `output/citation_results/`

| File | Description |
|---|---|
| `edges.parquet` | Internal citation edges |
| `nodes_raw.parquet` | Standardised node table |
| `nodes_metrics.parquet` | Node graph metrics |
| `country_flow.csv` | Country-to-country flow counts + shares |
| `frontier_stats.csv` | Frontier vs overall country share |
| `topic_coherence.csv` | Optional topic coherence metrics |
| `summary.json` | Pipeline summary + gap indicators |
| `figs/*.png` | Figures |


## §0  Setup & Config

Define paths, switches, and dependencies.


In [1]:
# Required packages (install manually if missing):
#   pip install pandas numpy networkx matplotlib tqdm pyarrow

import os
import re
import json
import math
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

try:
    import pyarrow  # noqa: F401
    PYARROW_AVAILABLE = True
except Exception:
    PYARROW_AVAILABLE = False

warnings.filterwarnings("ignore")

SEED = 42
rng = np.random.default_rng(SEED)

DATA_CSV = "data/dataCleanSCIE.csv"
TOPICS_CSV = None

OUTPUT_DIR = Path("output/citation_results")
FIGS_DIR = OUTPUT_DIR / "figs"

KEEP_EXTERNAL_NODES = False
PERM_B = 200
ROLLING_W = 5
EDGE_BATCH_SIZE = 200_000

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGS_DIR.mkdir(parents=True, exist_ok=True)

print("Config loaded")
print(f"DATA_CSV={DATA_CSV}")
print(f"TOPICS_CSV={TOPICS_CSV}")
print(f"OUTPUT_DIR={OUTPUT_DIR.resolve()}")
print(f"KEEP_EXTERNAL_NODES={KEEP_EXTERNAL_NODES}, PERM_B={PERM_B}, ROLLING_W={ROLLING_W}")
print(f"pyarrow available: {PYARROW_AVAILABLE}")


Config loaded
DATA_CSV=data/dataCleanSCIE.csv
TOPICS_CSV=None
OUTPUT_DIR=/Users/luoyiti/Project/catch-up/output/citation_results
KEEP_EXTERNAL_NODES=False, PERM_B=200, ROLLING_W=5
pyarrow available: True


/Users/luoyiti/Project/catch-up/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## §1  Load Data & Standardise Columns

Read CSV, standardise core fields, normalise DOI, deduplicate by `paper_id`, and export `nodes_raw.parquet`.


In [2]:
_RE_TRAIL_PUNCT = re.compile(r"[\s\.;,]+$")
_RE_DOI_URL_PREFIX = re.compile(r"^https?://(?:dx\.)?doi\.org/", re.IGNORECASE)

_CN_ALIASES = {
    "CHINA", "PEOPLES R CHINA", "PEOPLES R. CHINA", "PR CHINA", "PRC", "CN"
}
_US_ALIASES = {
    "UNITED STATES", "UNITED STATES OF AMERICA", "USA", "US", "U.S.A.", "U.S."
}


def normalize_text(x) -> Optional[str]:
    if pd.isna(x):
        return None
    s = str(x).strip()
    return s if s else None


def normalize_doi(x) -> Optional[str]:
    s = normalize_text(x)
    if not s:
        return None
    s = s.lower()
    s = _RE_DOI_URL_PREFIX.sub("", s)
    s = _RE_TRAIL_PUNCT.sub("", s)
    return s if s.startswith("10.") else None


def normalize_source(x) -> Optional[str]:
    s = normalize_text(x)
    if not s:
        return None
    s = re.sub(r"[^A-Za-z0-9\s]", " ", s.upper())
    s = re.sub(r"\s+", " ", s).strip()
    return s or None


def normalize_author_from_au(x) -> Optional[str]:
    s = normalize_text(x)
    if not s:
        return None
    first = s.split(";")[0].strip()
    first = first.split(",")[0].strip().upper()
    first = re.sub(r"\s+", " ", first)
    return first or None


def map_country_group(x) -> str:
    s = normalize_text(x)
    if not s:
        return "OTHER"
    su = s.upper()
    if su in _CN_ALIASES or "CHINA" in su:
        return "CN"
    if su in _US_ALIASES or "UNITED STATES" in su:
        return "US"
    return "OTHER"


print(f"Loading CSV: {DATA_CSV}")
raw_df = pd.read_csv(DATA_CSV, low_memory=False)
print(f"Raw shape: {raw_df.shape}")

col_map = {
    "UT": "paper_id",
    "DI": "doi",
    "PY": "year",
    "country": "country",
    "CR": "CR",
    "NR": "NR",
    "TC": "TC",
    "AU": "AU",
    "SO": "SO",
    "VL": "VL",
    "BP": "BP",
    "TI": "title",
}

df = raw_df.rename(columns={k: v for k, v in col_map.items() if k in raw_df.columns}).copy()

if "paper_id" not in df.columns:
    raise ValueError("Missing required column: UT")

for c in ["doi", "year", "country", "CR", "NR", "TC", "AU", "SO", "VL", "BP", "title"]:
    if c not in df.columns:
        df[c] = np.nan

before_dedup = len(df)
df["paper_id"] = df["paper_id"].astype(str).str.strip()
df = df[df["paper_id"].astype(str).str.len() > 0].copy()
df = df.drop_duplicates(subset=["paper_id"], keep="first").reset_index(drop=True)
after_dedup = len(df)

df["doi"] = df["doi"].apply(normalize_doi)
df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")
df["country_raw"] = df["country"].apply(lambda x: normalize_text(x) or "Unknown")
df["country_group"] = df["country_raw"].apply(map_country_group)
df["country"] = df["country_raw"]
df["title"] = df["title"].fillna("").astype(str)
df["NR"] = pd.to_numeric(df["NR"], errors="coerce")
df["TC"] = pd.to_numeric(df["TC"], errors="coerce")

df["author_key"] = df["AU"].apply(normalize_author_from_au)
df["source_key"] = df["SO"].apply(normalize_source)
df["volume_key"] = df["VL"].apply(lambda x: normalize_text(x).upper() if normalize_text(x) else None)
df["bp_key"] = df["BP"].apply(lambda x: normalize_text(x).upper() if normalize_text(x) else None)

nodes_raw_cols = [
    "paper_id", "doi", "year", "country", "country_group", "NR", "TC", "title", "CR",
    "author_key", "source_key", "volume_key", "bp_key",
]
nodes_raw = df[nodes_raw_cols].copy()
nodes_raw.to_parquet(OUTPUT_DIR / "nodes_raw.parquet", index=False)

print(f"Dedup by paper_id: {before_dedup:,} -> {after_dedup:,}")
print(f"Year range: {nodes_raw['year'].min()} - {nodes_raw['year'].max()}")
print(f"DOI non-null: {nodes_raw['doi'].notna().sum():,}/{len(nodes_raw):,}")
print("Country group counts:", nodes_raw["country_group"].value_counts(dropna=False).to_dict())
print(f"Saved nodes_raw.parquet: {OUTPUT_DIR / 'nodes_raw.parquet'}")


Loading CSV: data/dataCleanSCIE.csv
Raw shape: (25794, 41)
Dedup by paper_id: 25,794 -> 25,794
Year range: 1990 - 2026
DOI non-null: 24,018/25,794
Saved nodes_raw.parquet: output/citation_results/nodes_raw.parquet


## §2  Parse CR & Build Internal Citation Edges

Build DOI/key maps, parse references, match to internal papers, stream-write edges, and export diagnostics.


In [3]:
_RE_DOI = re.compile(r"(?i)\b(10\.\d{4,9}/[-._;()/:A-Z0-9+#]+)")
_RE_YEAR = re.compile(r"\b(18\d{2}|19\d{2}|20\d{2})\b")
_RE_VOL = re.compile(r"\bV([A-Z0-9]+)\b", re.IGNORECASE)
_RE_BP = re.compile(r"\bP([A-Z0-9]+)\b", re.IGNORECASE)
_AMBIG = "__AMBIG__"


def split_cr(cr_str) -> List[str]:
    if pd.isna(cr_str):
        return []
    s = str(cr_str).strip()
    if not s:
        return []
    return [x.strip() for x in s.split("; ") if x and x.strip()]


def extract_dois(ref_str) -> List[str]:
    if not ref_str:
        return []
    found = _RE_DOI.findall(str(ref_str))
    out = []
    seen = set()
    for d in found:
        nd = normalize_doi(d)
        if nd and nd not in seen:
            out.append(nd)
            seen.add(nd)
    return out


def _ref_first_author(ref_str: str) -> Optional[str]:
    parts = [p.strip() for p in str(ref_str).split(",")]
    if not parts or not parts[0]:
        return None
    tokens = parts[0].upper().split()
    if not tokens:
        return None
    if len(tokens) == 1:
        return tokens[0]
    return " ".join(tokens[:-1])


def _ref_source(ref_str: str, year_val: Optional[int]) -> Optional[str]:
    parts = [p.strip() for p in str(ref_str).split(",")]
    if year_val is None:
        return None
    y = str(year_val)
    for i, p in enumerate(parts):
        if y in p and i + 1 < len(parts):
            return normalize_source(parts[i + 1])
    return None


def _ref_year(ref_str: str) -> Optional[int]:
    m = _RE_YEAR.search(str(ref_str))
    return int(m.group(1)) if m else None


def _ref_volume(ref_str: str) -> Optional[str]:
    m = _RE_VOL.search(str(ref_str))
    return m.group(1).upper() if m else None


def _ref_bp(ref_str: str) -> Optional[str]:
    m = _RE_BP.search(str(ref_str))
    return m.group(1).upper() if m else None


def _build_key(author: Optional[str], year: Optional[int], source: Optional[str], vol: Optional[str], bp: Optional[str]):
    if not author or year is None or not source or not vol or not bp:
        return None
    return (author, int(year), source, vol, bp)


def build_maps(df_nodes: pd.DataFrame) -> Tuple[Dict[str, str], Dict[Tuple[str, int, str, str, str], str]]:
    doi2pid: Dict[str, str] = {}
    key2pid: Dict[Tuple[str, int, str, str, str], str] = {}

    for row in tqdm(df_nodes.itertuples(index=False), total=len(df_nodes), desc="build_maps"):
        pid = row.paper_id

        d = row.doi
        if isinstance(d, str) and d:
            if d in doi2pid and doi2pid[d] != pid:
                doi2pid[d] = _AMBIG
            else:
                doi2pid[d] = pid

        key = _build_key(row.author_key, None if pd.isna(row.year) else int(row.year), row.source_key, row.volume_key, row.bp_key)
        if key is not None:
            if key in key2pid and key2pid[key] != pid:
                key2pid[key] = _AMBIG
            else:
                key2pid[key] = pid

    return doi2pid, key2pid


def match_ref_to_pid(ref_str: str, doi2pid: Dict[str, str], key2pid: Dict[Tuple[str, int, str, str, str], str]) -> Tuple[Optional[str], str, Optional[str]]:
    dois = extract_dois(ref_str)
    for d in dois:
        hit = doi2pid.get(d)
        if hit and hit != _AMBIG:
            return hit, "doi", d

    year_v = _ref_year(ref_str)
    author_v = _ref_first_author(ref_str)
    source_v = _ref_source(ref_str, year_v)
    vol_v = _ref_volume(ref_str)
    bp_v = _ref_bp(ref_str)
    key = _build_key(author_v, year_v, source_v, vol_v, bp_v)
    if key is not None:
        hit = key2pid.get(key)
        if hit and hit != _AMBIG:
            return hit, "key", None

    return None, "none", (dois[0] if dois else None)


def build_edges(
    df_nodes: pd.DataFrame,
    doi2pid: Dict[str, str],
    key2pid: Dict[Tuple[str, int, str, str, str], str],
    output_dir: Path,
    keep_external: bool = False,
    batch_size: int = 200_000,
) -> Tuple[pd.DataFrame, dict]:
    edge_cols = [
        "src_paper_id", "dst_paper_id", "match_type", "matched_doi", "ref_raw",
        "src_year", "dst_year", "src_country", "dst_country",
    ]

    if "country_group" not in df_nodes.columns:
        df_nodes = df_nodes.copy()
        df_nodes["country_group"] = df_nodes["country"].apply(map_country_group)

    pid_set = set(df_nodes["paper_id"].tolist())
    meta = df_nodes.set_index("paper_id")[["year", "country", "country_group"]].to_dict("index")

    tmp_dir = output_dir / "_tmp_edges"
    if tmp_dir.exists():
        for p in tmp_dir.glob("*.parquet"):
            p.unlink()
    tmp_dir.mkdir(parents=True, exist_ok=True)

    part_files: List[Path] = []
    buffer: List[dict] = []
    part_idx = 0

    total_ref_items = 0
    refs_with_doi = 0
    internal_hits = 0
    external_or_unmatched = 0
    match_type_counts = {"doi": 0, "key": 0, "none": 0}

    for row in tqdm(df_nodes.itertuples(index=False), total=len(df_nodes), desc="build_edges"):
        src = row.paper_id
        src_year = None if pd.isna(row.year) else int(row.year)
        src_country = row.country if isinstance(row.country, str) and row.country else "Unknown"

        for ref_raw in split_cr(row.CR):
            total_ref_items += 1
            doi_candidates = extract_dois(ref_raw)
            if doi_candidates:
                refs_with_doi += 1

            dst, mtype, mdoi = match_ref_to_pid(ref_raw, doi2pid, key2pid)
            match_type_counts[mtype] = match_type_counts.get(mtype, 0) + 1

            if dst in pid_set and dst != src:
                internal_hits += 1
                dst_meta = meta.get(dst, {})
                dst_year = dst_meta.get("year")
                if pd.isna(dst_year):
                    dst_year = None
                else:
                    dst_year = int(dst_year)
                dst_country = dst_meta.get("country", "Unknown")
                rec = {
                    "src_paper_id": src,
                    "dst_paper_id": dst,
                    "match_type": mtype,
                    "matched_doi": mdoi,
                    "ref_raw": ref_raw,
                    "src_year": src_year,
                    "dst_year": dst_year,
                    "src_country": src_country,
                    "dst_country": dst_country,
                }
                buffer.append(rec)
            else:
                external_or_unmatched += 1
                if keep_external:
                    buffer.append({
                        "src_paper_id": src,
                        "dst_paper_id": None,
                        "match_type": "external",
                        "matched_doi": mdoi,
                        "ref_raw": ref_raw,
                        "src_year": src_year,
                        "dst_year": None,
                        "src_country": src_country,
                        "dst_country": "External",
                    })

            if len(buffer) >= batch_size:
                part_path = tmp_dir / f"edges_part_{part_idx:05d}.parquet"
                pd.DataFrame(buffer, columns=edge_cols).to_parquet(part_path, index=False)
                part_files.append(part_path)
                buffer.clear()
                part_idx += 1

    if buffer:
        part_path = tmp_dir / f"edges_part_{part_idx:05d}.parquet"
        pd.DataFrame(buffer, columns=edge_cols).to_parquet(part_path, index=False)
        part_files.append(part_path)
        buffer.clear()

    if part_files:
        edges_df = pd.concat([pd.read_parquet(p) for p in part_files], ignore_index=True)
        edges_df = edges_df[edge_cols]
    else:
        edges_df = pd.DataFrame(columns=edge_cols)

    edges_path = output_dir / "edges.parquet"
    edges_df.to_parquet(edges_path, index=False)

    for p in part_files:
        p.unlink(missing_ok=True)
    try:
        tmp_dir.rmdir()
    except Exception:
        pass

    diagnostics = {
        "total_ref_items": int(total_ref_items),
        "refs_with_doi": int(refs_with_doi),
        "doi_extract_rate": float(refs_with_doi / total_ref_items) if total_ref_items else 0.0,
        "internal_hits": int(internal_hits),
        "internal_hit_rate": float(internal_hits / total_ref_items) if total_ref_items else 0.0,
        "external_or_unmatched": int(external_or_unmatched),
        "external_ratio": float(external_or_unmatched / total_ref_items) if total_ref_items else 0.0,
        "match_type_counts": {k: int(v) for k, v in match_type_counts.items()},
        "keep_external_nodes": bool(keep_external),
    }

    with open(output_dir / "diagnostics.json", "w", encoding="utf-8") as f:
        json.dump(diagnostics, f, indent=2)

    return edges_df, diagnostics


doi2pid, key2pid = build_maps(nodes_raw)
edges_df, edge_diagnostics = build_edges(
    nodes_raw,
    doi2pid,
    key2pid,
    OUTPUT_DIR,
    keep_external=KEEP_EXTERNAL_NODES,
    batch_size=EDGE_BATCH_SIZE,
)

print(f"Saved edges.parquet ({len(edges_df):,} rows)")
print("Saved diagnostics.json")
print(edge_diagnostics)


build_edges: 100%|██████████| 25794/25794 [00:04<00:00, 5229.34it/s]


Saved edges.parquet (49,089 rows)
Saved diagnostics.json
{'total_ref_items': 801194, 'refs_with_doi': 538386, 'doi_extract_rate': 0.6719795704910421, 'internal_hits': 49089, 'internal_hit_rate': 0.061269804816311654, 'external_or_unmatched': 752105, 'external_ratio': 0.9387301951836884, 'match_type_counts': {'doi': 49071, 'key': 55, 'none': 752068}, 'keep_external_nodes': False}


## §3  Build Graph & Node Metrics

Construct `networkx.DiGraph`, compute in/out degree and PageRank, and export `nodes_metrics.parquet`.


In [4]:
def build_graph_and_metrics(nodes_df: pd.DataFrame, edges_df: pd.DataFrame) -> Tuple[nx.DiGraph, pd.DataFrame]:
    G = nx.DiGraph()
    G.add_nodes_from(nodes_df["paper_id"].tolist())

    if len(edges_df):
        edge_counts = (
            edges_df.groupby(["src_paper_id", "dst_paper_id"], as_index=False)
            .size()
            .rename(columns={"size": "weight"})
        )
        G.add_weighted_edges_from(edge_counts[["src_paper_id", "dst_paper_id", "weight"]].itertuples(index=False, name=None))

    in_deg = dict(G.in_degree())
    out_deg = dict(G.out_degree())

    try:
        pagerank = nx.pagerank(G, alpha=0.85, max_iter=200)
    except nx.PowerIterationFailedConvergence:
        try:
            pagerank = nx.pagerank(G, alpha=0.85, max_iter=1000)
        except Exception:
            pagerank = {n: 0.0 for n in G.nodes()}

    metrics = nodes_df[["paper_id", "doi", "year", "country"]].copy()
    metrics["in_degree"] = metrics["paper_id"].map(in_deg).fillna(0).astype(int)
    metrics["out_degree"] = metrics["paper_id"].map(out_deg).fillna(0).astype(int)
    metrics["pagerank"] = metrics["paper_id"].map(pagerank).fillna(0.0).astype(float)

    return G, metrics


citation_graph, nodes_metrics = build_graph_and_metrics(nodes_raw, edges_df)
nodes_metrics.to_parquet(OUTPUT_DIR / "nodes_metrics.parquet", index=False)

print(f"Graph: nodes={citation_graph.number_of_nodes():,}, edges={citation_graph.number_of_edges():,}")
print(f"Saved nodes_metrics.parquet ({len(nodes_metrics):,} rows)")


Graph: nodes=25,794, edges=49,084
Saved nodes_metrics.parquet (25,794 rows)


## §4  Cross-Country Citation Structure

Build country flow matrix (`count` + `share`), export CSV, and draw heatmap + CN↔US over-time shares.


In [5]:
def compute_country_flow(edges_df: pd.DataFrame, output_dir: Path, figs_dir: Path, rolling_w: int = 5):
    if len(edges_df) == 0:
        flow_df = pd.DataFrame(columns=["src_country", "dst_country", "count", "share_of_src", "share_global"])
    else:
        flow_df = (
            edges_df.groupby(["src_country", "dst_country"], dropna=False)
            .size()
            .reset_index(name="count")
        )
        flow_df["src_country"] = flow_df["src_country"].fillna("Unknown")
        flow_df["dst_country"] = flow_df["dst_country"].fillna("Unknown")
        src_tot = flow_df.groupby("src_country")["count"].transform("sum")
        global_tot = flow_df["count"].sum()
        flow_df["share_of_src"] = np.where(src_tot > 0, flow_df["count"] / src_tot, 0.0)
        flow_df["share_global"] = np.where(global_tot > 0, flow_df["count"] / global_tot, 0.0)

    flow_df.to_csv(output_dir / "country_flow.csv", index=False)

    if len(flow_df):
        top_countries = (
            pd.concat([flow_df["src_country"], flow_df["dst_country"]])
            .value_counts()
            .head(12)
            .index
            .tolist()
        )
        plot_df = flow_df[flow_df["src_country"].isin(top_countries) & flow_df["dst_country"].isin(top_countries)].copy()
        mat = plot_df.pivot(index="src_country", columns="dst_country", values="count").fillna(0.0)
        mat = mat.reindex(index=top_countries, columns=top_countries, fill_value=0.0)
    else:
        top_countries = ["CN", "US"]
        mat = pd.DataFrame([[0.0, 0.0], [0.0, 0.0]], index=top_countries, columns=top_countries)

    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(mat.values, aspect="auto", cmap="Blues")
    ax.set_xticks(range(len(mat.columns)))
    ax.set_xticklabels(mat.columns, rotation=45, ha="right")
    ax.set_yticks(range(len(mat.index)))
    ax.set_yticklabels(mat.index)
    ax.set_title("Country Citation Flow Heatmap (count)")
    vmax = float(np.nanmax(mat.values)) if mat.size else 0.0
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            v = int(mat.values[i, j])
            ax.text(j, i, f"{v}", ha="center", va="center", fontsize=8,
                    color="white" if vmax > 0 and v > vmax * 0.6 else "black")
    plt.colorbar(im, ax=ax, shrink=0.8)
    plt.tight_layout()
    fig.savefig(figs_dir / "flow_heatmap.png", dpi=150)
    plt.show()
    plt.close(fig)

    made_time_fig = False
    cn_to_us_count = us_to_cn_count = cn_total_out = us_total_out = 0
    if len(edges_df):
        grp = edges_df[["src_country", "dst_country", "src_year"]].copy()
        grp["src_group"] = grp["src_country"].apply(map_country_group)
        grp["dst_group"] = grp["dst_country"].apply(map_country_group)

        cn_to_us_count = int(((grp["src_group"] == "CN") & (grp["dst_group"] == "US")).sum())
        us_to_cn_count = int(((grp["src_group"] == "US") & (grp["dst_group"] == "CN")).sum())
        cn_total_out = int((grp["src_group"] == "CN").sum())
        us_total_out = int((grp["src_group"] == "US").sum())

        if grp["src_year"].notna().any():
            ts = grp.copy()
            ts["src_year"] = pd.to_numeric(ts["src_year"], errors="coerce")
            ts = ts.dropna(subset=["src_year"]).copy()
            ts["src_year"] = ts["src_year"].astype(int)

            cn_all = ts[ts["src_group"] == "CN"].groupby("src_year").size().rename("cn_total")
            us_all = ts[ts["src_group"] == "US"].groupby("src_year").size().rename("us_total")

            cn_us = ts[(ts["src_group"] == "CN") & (ts["dst_group"] == "US")].groupby("src_year").size().rename("cn_to_us")
            us_cn = ts[(ts["src_group"] == "US") & (ts["dst_group"] == "CN")].groupby("src_year").size().rename("us_to_cn")

            year_df = pd.concat([cn_all, us_all, cn_us, us_cn], axis=1).fillna(0).sort_index()
            year_df["cn_to_us_share"] = np.where(year_df["cn_total"] > 0, year_df["cn_to_us"] / year_df["cn_total"], np.nan)
            year_df["us_to_cn_share"] = np.where(year_df["us_total"] > 0, year_df["us_to_cn"] / year_df["us_total"], np.nan)

            year_df["cn_to_us_share_roll"] = year_df["cn_to_us_share"].rolling(rolling_w, min_periods=1).mean()
            year_df["us_to_cn_share_roll"] = year_df["us_to_cn_share"].rolling(rolling_w, min_periods=1).mean()

            if len(year_df):
                fig, ax = plt.subplots(figsize=(10, 5))
                ax.plot(year_df.index, year_df["cn_to_us_share_roll"], label="CN->US share (rolling)", color="tab:red")
                ax.plot(year_df.index, year_df["us_to_cn_share_roll"], label="US->CN share (rolling)", color="tab:blue")
                ax.set_xlabel("Year")
                ax.set_ylabel("Share")
                ax.set_title(f"CN<->US Citation Share Over Time (rolling={rolling_w})")
                ax.grid(True, alpha=0.3)
                ax.legend()
                ax.set_ylim(0, 1)
                plt.tight_layout()
                fig.savefig(figs_dir / "flow_over_time.png", dpi=150)
                plt.show()
                plt.close(fig)
                made_time_fig = True

    metrics = {
        "cn_to_us_count": int(cn_to_us_count),
        "us_to_cn_count": int(us_to_cn_count),
        "cn_total_out": int(cn_total_out),
        "us_total_out": int(us_total_out),
        "cn_to_us_dependency": (cn_to_us_count / cn_total_out) if cn_total_out else 0.0,
        "us_to_cn_dependency": (us_to_cn_count / us_total_out) if us_total_out else 0.0,
        "made_flow_over_time_fig": bool(made_time_fig),
    }

    return flow_df, mat, metrics


country_flow_df, country_flow_matrix, flow_metrics = compute_country_flow(edges_df, OUTPUT_DIR, FIGS_DIR, rolling_w=ROLLING_W)
print(f"Saved country_flow.csv ({len(country_flow_df):,} rows)")
print("Saved figs/flow_heatmap.png")
if flow_metrics["made_flow_over_time_fig"]:
    print("Saved figs/flow_over_time.png")


Saved country_flow.csv (1,806 rows)
Saved figs/flow_heatmap.png


## §5  Topic–Citation Coherence（可选，若 topic 存在）

If `TOPICS_CSV` exists and columns are detectable, compute intra-topic citation ratio with permutation baseline.


In [6]:
def _pick_col_by_candidates(df: pd.DataFrame, candidates: List[str]) -> Optional[str]:
    lower_map = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    return None


def compute_topic_coherence_optional(
    edges_df: pd.DataFrame,
    topics_csv: Optional[str],
    perm_b: int,
    output_dir: Path,
    figs_dir: Path,
    seed: int = 42,
):
    if not topics_csv:
        print("TOPICS_CSV is None; skip topic coherence.")
        return None

    path = Path(topics_csv)
    if not path.exists():
        print(f"TOPICS_CSV not found: {topics_csv}; skip topic coherence.")
        return None

    tdf = pd.read_csv(path, low_memory=False)
    id_col = _pick_col_by_candidates(tdf, ["UT", "paper_id", "id"])
    topic_col = _pick_col_by_candidates(tdf, ["topic", "Topic", "topic_id", "cluster"])

    if id_col is None or topic_col is None:
        print("Cannot detect topic id/topic columns; skip topic coherence.")
        return None

    topic_map_df = tdf[[id_col, topic_col]].copy()
    topic_map_df.columns = ["paper_id", "topic"]
    topic_map_df["paper_id"] = topic_map_df["paper_id"].astype(str).str.strip()
    topic_map_df["topic"] = topic_map_df["topic"].astype(str).str.strip()
    topic_map_df = topic_map_df[(topic_map_df["paper_id"].str.len() > 0) & (topic_map_df["topic"].str.len() > 0)]
    topic_map_df = topic_map_df.drop_duplicates(subset=["paper_id"], keep="first").reset_index(drop=True)

    if len(topic_map_df) == 0:
        print("Topic map empty after cleaning; skip topic coherence.")
        return None

    pid_to_pos = {pid: i for i, pid in enumerate(topic_map_df["paper_id"].tolist())}
    labels = topic_map_df["topic"].to_numpy()

    if len(edges_df) == 0:
        print("No edges available; skip topic coherence.")
        return None

    src = edges_df["src_paper_id"].astype(str).to_numpy()
    dst = edges_df["dst_paper_id"].astype(str).to_numpy()

    valid_mask = np.array([(s in pid_to_pos) and (d in pid_to_pos) for s, d in zip(src, dst)], dtype=bool)
    if not valid_mask.any():
        print("No edges with both endpoints in topic map; skip topic coherence.")
        return None

    src_pos = np.array([pid_to_pos[s] for s in src[valid_mask]], dtype=int)
    dst_pos = np.array([pid_to_pos[d] for d in dst[valid_mask]], dtype=int)

    observed = float(np.mean(labels[src_pos] == labels[dst_pos]))

    rng_local = np.random.default_rng(seed)
    perm_vals = np.zeros(perm_b, dtype=float)
    for i in tqdm(range(perm_b), desc="topic_permutation"):
        perm = rng_local.permutation(len(labels))
        perm_vals[i] = float(np.mean(labels[perm[src_pos]] == labels[perm[dst_pos]]))

    perm_mean = float(np.mean(perm_vals))
    perm_p95 = float(np.quantile(perm_vals, 0.95))
    percentile = float((perm_vals <= observed).mean())

    out_df = pd.DataFrame([
        {
            "observed": observed,
            "perm_mean": perm_mean,
            "perm_p95": perm_p95,
            "percentile": percentile,
            "n_edges_with_topics": int(len(src_pos)),
            "perm_b": int(perm_b),
        }
    ])
    out_df.to_csv(output_dir / "topic_coherence.csv", index=False)

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.hist(perm_vals, bins=30, alpha=0.75, color="steelblue", edgecolor="white")
    ax.axvline(observed, color="crimson", linestyle="--", linewidth=2, label=f"observed={observed:.4f}")
    ax.axvline(perm_mean, color="black", linestyle=":", linewidth=1.5, label=f"perm_mean={perm_mean:.4f}")
    ax.set_title("Topic-Citation Coherence vs Permutation Baseline")
    ax.set_xlabel("Intra-topic edge ratio")
    ax.set_ylabel("Frequency")
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.tight_layout()
    fig.savefig(figs_dir / "topic_coherence_perm.png", dpi=150)
    plt.show()
    plt.close(fig)

    return {
        "observed": observed,
        "perm_mean": perm_mean,
        "perm_p95": perm_p95,
        "percentile": percentile,
        "n_edges_with_topics": int(len(src_pos)),
        "perm_b": int(perm_b),
    }


topic_result = compute_topic_coherence_optional(
    edges_df=edges_df,
    topics_csv=TOPICS_CSV,
    perm_b=PERM_B,
    output_dir=OUTPUT_DIR,
    figs_dir=FIGS_DIR,
    seed=SEED,
)

if topic_result is not None:
    print("Saved topic_coherence.csv")
    print("Saved figs/topic_coherence_perm.png")
    print(topic_result)


TOPICS_CSV is None; skip topic coherence.


## §6  Core/Frontier Gap（网络前沿）

Define frontier by PageRank top `min(500, max(1, ceil(0.01*N)))`, export frontier stats, and draw CN/US share comparison.


In [7]:
def compute_frontier_stats(nodes_metrics_df: pd.DataFrame, output_dir: Path, figs_dir: Path):
    if len(nodes_metrics_df) == 0:
        frontier_df = pd.DataFrame(columns=["country", "is_frontier", "count", "share"])
        frontier_df.to_csv(output_dir / "frontier_stats.csv", index=False)

        fig, ax = plt.subplots(figsize=(6, 4))
        ax.bar(["CN", "US"], [0, 0], color=["tab:red", "tab:blue"])
        ax.set_ylim(0, 1)
        ax.set_title("Frontier Share (empty graph)")
        plt.tight_layout()
        fig.savefig(figs_dir / "frontier_share.png", dpi=150)
        plt.show()
        plt.close(fig)

        return frontier_df, {
            "frontier_top_n": 0,
            "cn_frontier_share": 0.0,
            "us_frontier_share": 0.0,
            "cn_overall_share": 0.0,
            "us_overall_share": 0.0,
            "frontier_share_gap": 0.0,
        }

    n = len(nodes_metrics_df)
    top_n = min(500, max(1, int(math.ceil(0.01 * n))))

    ranked = nodes_metrics_df.sort_values("pagerank", ascending=False).reset_index(drop=True)
    frontier_ids = set(ranked.head(top_n)["paper_id"].tolist())

    tmp = nodes_metrics_df[["paper_id", "country"]].copy()
    tmp["country"] = tmp["country"].apply(map_country_group)
    tmp["is_frontier"] = tmp["paper_id"].isin(frontier_ids)

    stats = tmp.groupby(["country", "is_frontier"]).size().reset_index(name="count")
    group_total = stats.groupby("is_frontier")["count"].transform("sum")
    stats["share"] = np.where(group_total > 0, stats["count"] / group_total, 0.0)
    stats.to_csv(output_dir / "frontier_stats.csv", index=False)

    def _share(df, country, frontier_flag):
        m = df[(df["country"] == country) & (df["is_frontier"] == frontier_flag)]
        if len(m) == 0:
            return 0.0
        return float(m["share"].iloc[0])

    cn_frontier_share = _share(stats, "CN", True)
    us_frontier_share = _share(stats, "US", True)

    overall = tmp.groupby("country").size()
    cn_overall_share = float(overall.get("CN", 0) / len(tmp)) if len(tmp) else 0.0
    us_overall_share = float(overall.get("US", 0) / len(tmp)) if len(tmp) else 0.0

    labels = ["CN", "US"]
    x = np.arange(len(labels))
    w = 0.35
    frontier_vals = [cn_frontier_share, us_frontier_share]
    overall_vals = [cn_overall_share, us_overall_share]

    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.bar(x - w / 2, overall_vals, width=w, label="Overall", color=["#f4a3a3", "#a8c7ff"])
    ax.bar(x + w / 2, frontier_vals, width=w, label="Frontier", color=["tab:red", "tab:blue"])
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_ylim(0, 1)
    ax.set_ylabel("Share")
    ax.set_title(f"Country Share in Frontier (top_n={top_n})")
    ax.grid(True, axis="y", alpha=0.3)
    ax.legend()
    plt.tight_layout()
    fig.savefig(figs_dir / "frontier_share.png", dpi=150)
    plt.show()
    plt.close(fig)

    metrics = {
        "frontier_top_n": int(top_n),
        "cn_frontier_share": float(cn_frontier_share),
        "us_frontier_share": float(us_frontier_share),
        "cn_overall_share": float(cn_overall_share),
        "us_overall_share": float(us_overall_share),
        "frontier_share_gap": float(cn_frontier_share - us_frontier_share),
    }
    return stats, metrics


frontier_stats_df, frontier_metrics = compute_frontier_stats(nodes_metrics, OUTPUT_DIR, FIGS_DIR)
print(f"Saved frontier_stats.csv ({len(frontier_stats_df):,} rows)")
print("Saved figs/frontier_share.png")
print(frontier_metrics)


Saved frontier_stats.csv (136 rows)
Saved figs/frontier_share.png
{'frontier_top_n': 258, 'cn_frontier_share': 0.0, 'us_frontier_share': 0.0, 'cn_overall_share': 0.0, 'us_overall_share': 0.0, 'frontier_share_gap': 0.0}


## §7  Summary JSON & Manifest

Aggregate key metrics and write `summary.json`, then print product manifest.


In [8]:
def _json_default(obj):
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.ndarray,)):
        return obj.tolist()
    try:
        if pd.isna(obj):
            return None
    except Exception:
        pass
    return str(obj)


def _largest_weak_component_ratio(G: nx.DiGraph) -> float:
    if G.number_of_nodes() == 0:
        return 0.0
    comps = list(nx.weakly_connected_components(G))
    if not comps:
        return 0.0
    max_sz = max(len(c) for c in comps)
    return float(max_sz / G.number_of_nodes())


def build_summary(
    nodes_df: pd.DataFrame,
    edges_df: pd.DataFrame,
    graph_obj: nx.DiGraph,
    diagnostics: dict,
    flow_metrics: dict,
    frontier_metrics: dict,
    topic_result: Optional[dict],
):
    cn_us = float(flow_metrics.get("cn_to_us_count", 0))
    us_cn = float(flow_metrics.get("us_to_cn_count", 0))
    eps = 1e-12

    summary = {
        "config": {
            "data_csv": DATA_CSV,
            "topics_csv": TOPICS_CSV,
            "keep_external_nodes": KEEP_EXTERNAL_NODES,
            "perm_b": PERM_B,
            "rolling_w": ROLLING_W,
            "edge_batch_size": EDGE_BATCH_SIZE,
            "seed": SEED,
        },
        "graph": {
            "num_nodes": int(len(nodes_df)),
            "num_edges": int(len(edges_df)),
            "largest_weak_component_ratio": _largest_weak_component_ratio(graph_obj),
        },
        "diagnostics": diagnostics,
        "gap_metrics": {
            "cn_to_us_dependency": float(flow_metrics.get("cn_to_us_dependency", 0.0)),
            "us_to_cn_dependency": float(flow_metrics.get("us_to_cn_dependency", 0.0)),
            "asymmetry_index": float((cn_us - us_cn) / (cn_us + us_cn + eps)),
            "frontier_share_gap": float(frontier_metrics.get("frontier_share_gap", 0.0)),
        },
        "flow_metrics": flow_metrics,
        "frontier_metrics": frontier_metrics,
        "topic_coherence": topic_result if topic_result is not None else None,
    }
    return summary


summary = build_summary(
    nodes_df=nodes_metrics,
    edges_df=edges_df,
    graph_obj=citation_graph,
    diagnostics=edge_diagnostics,
    flow_metrics=flow_metrics,
    frontier_metrics=frontier_metrics,
    topic_result=topic_result,
)

with open(OUTPUT_DIR / "summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, default=_json_default)

print("Saved summary.json")

print("\n" + "=" * 62)
print("Paper Citation Graph  --  Product Manifest")
print(OUTPUT_DIR.resolve())
print("=" * 62)
for root, dirs, files in os.walk(OUTPUT_DIR):
    dirs.sort()
    files.sort()
    level = str(root).replace(str(OUTPUT_DIR), "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{Path(root).name}/")
    for fn in files:
        fp = Path(root) / fn
        sz = fp.stat().st_size
        if sz >= 1024 * 1024:
            disp = f"{sz / (1024 * 1024):.1f} MB"
        else:
            disp = f"{sz / 1024:.1f} KB"
        print(f"{indent}  {fn}  ({disp})")


Saved summary.json

Paper Citation Graph  --  Product Manifest
/Users/luoyiti/Project/catch-up/output/citation_results
citation_results/
  country_flow.csv  (111.9 KB)
  diagnostics.json  (0.4 KB)
  edges.parquet  (1.2 MB)
  frontier_stats.csv  (5.3 KB)
  nodes_metrics.parquet  (711.5 KB)
  nodes_raw.parquet  (32.7 MB)
  summary.json  (1.3 KB)
  figs/
    flow_heatmap.png  (135.7 KB)
    flow_over_time.png  (36.1 KB)
    frontier_share.png  (22.4 KB)
